In [126]:
import numpy as np
import sympy as sp
A = np.array([
    [1.0,  -2.0, -1.0,  1.0],
    [2.0,  -3.0, 1.0,  6.0],
    [3.0,  -5.0,  0.0,  7.0],
    [1.0, 0.0, 5.0, 9.0]
], dtype=float)

In [127]:
def swap_rows(A, i, j):
    temp = A[i].copy()
    A[i] = A[j]
    A[j] = temp

In [128]:
def multiply_row(A, i, k):
    A[i] = A[i] * k

In [129]:
def add_row_multiple(A, dest_row, src_row, k):
    A[dest_row] = A[dest_row] - k*A[src_row]

In [130]:
def is_zero(val, tol=1e-9):
    return abs(val) < tol

In [131]:
def gauss_elimination(A):
    cur_row = 0
    n, m = A.shape
    for j in range (0, m-1):
        if cur_row >= n:
            break

        max_val = 0 
        pivot_row = cur_row
        for i in range (cur_row, n):
            if abs(A[i, j]) > max_val:
                max_val = abs(A[i, j])
                pivot_row = i
        
        if is_zero(max_val):
            continue
        
        swap_rows(A, cur_row, pivot_row)
        
        multiply_row(A, cur_row, 1/A[cur_row, j])

        for k in range (cur_row+1, n):
            val = float(A[k ,j])
            add_row_multiple(A, k, cur_row, val)

        cur_row += 1

    return A

In [132]:
print("--- MA TRẬN BAN ĐẦU ---")
print(A)
print("\n-----------------------")

# 2. Gọi hàm Gauss để biến đổi ma trận A
A_bac_thang = gauss_elimination(A)

print("--- MA TRẬN BẬC THANG THU ĐƯỢC ---")
print(A_bac_thang)

--- MA TRẬN BAN ĐẦU ---
[[ 1. -2. -1.  1.]
 [ 2. -3.  1.  6.]
 [ 3. -5.  0.  7.]
 [ 1.  0.  5.  9.]]

-----------------------
--- MA TRẬN BẬC THANG THU ĐƯỢC ---
[[ 1.00000000e+00 -1.66666667e+00  0.00000000e+00  2.33333333e+00]
 [ 0.00000000e+00  1.00000000e+00  3.00000000e+00  4.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  6.66133815e-16  1.11022302e-15]
 [ 0.00000000e+00  0.00000000e+00  7.77156117e-16  1.55431223e-15]]


In [133]:
def back_substitution(A):
    n, m = A.shape
    num_variables = m - 1

    # kiem tra pt vo nghiem
    for i in range (n-1, -1, -1):
        all_variables_zero = np.all([is_zero(A[i, j]) for j in range (num_variables)])
        left_zero = not is_zero(A[i, m-1])

        if (all_variables_zero and left_zero):
            print("Hệ vô nghiệm")
            return None
    
    # giai nghiem
    x_sym = [sp.symbols(f'x_{i+1}') for i in range(num_variables)]
    
    pivot_columns = []
    for i in range(n):
        for j in range(num_variables):
            if not is_zero(A[i, j]):
                pivot_columns.append(j)
                break

    num_pivots = len(pivot_columns)
    for i in range(num_pivots - 1, -1, -1):
        j = pivot_columns[i]

        left_sum = sum([A[i, k] * x_sym[k] for k in range(j+1, num_variables)])

        x_sym[j] = sp.simplify(A[i, m-1] - left_sum)


    return x_sym

In [134]:
solution = back_substitution(A_bac_thang)
print(solution)

[9.0 - 5.0*x_3, 4.0 - 3.0*x_3, x_3]


In [135]:
def load_and_solve_matrices(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    current_matrix_lines = []
    matrix_name = "Chưa rõ"
    
    for line in lines:
        line = line.strip()
        
        if not line:
            continue
            
        if line.startswith("#"):
            matrix_name = line.replace("#", "").strip()
            continue

        if line == "---":
            if current_matrix_lines:
                # 1. Chuyển thành mảng numpy kiểu float
                A = np.array(current_matrix_lines, dtype=float)
                
                print(f"=== {matrix_name} ===")
                # 2. Gọi hàm giải và hứng kết quả trả về vào biến result
                result = gauss_elimination(A)
                result = back_substitution(result)
                
                # 3. In trực tiếp kết quả ma trận bậc thang ra màn hình
                print("Ma trận bậc thang kết quả:")
                print(result)
                print("-" * 30)
               
                current_matrix_lines = []
            continue
            
        row_data = [float(x) for x in line.split()]
        current_matrix_lines.append(row_data)
        
    # Xử lý cho ma trận cuối cùng ở đáy file (nơi không có dấu ---)
    if current_matrix_lines:
        A = np.array(current_matrix_lines, dtype=float)
        print(f"=== {matrix_name} ===")
        
        result = gauss_elimination(A)
        result = back_substitution(result)
        print("Ma trận bậc thang kết quả:")
        print(result)
        print("-" * 30)

if __name__ == "__main__":
    # Đảm bảo file test.txt của bạn đã điền đúng chuẩn format
    load_and_solve_matrices("test.txt")

=== Bai 1 ===
Ma trận bậc thang kết quả:
[4.00000000000000, -3.00000000000000, -0.999999999999998]
------------------------------
=== Bai 2 ===
Ma trận bậc thang kết quả:
[9.0 - 5.0*x_3, 4.0 - 3.0*x_3, x_3]
------------------------------
=== Bai 3 ===
Ma trận bậc thang kết quả:
[2.00000000000000, 3.00000000000000, -2.00000000000000, -0.999999999999998]
------------------------------
=== Bai 4 ===
Hệ vô nghiệm
Ma trận bậc thang kết quả:
None
------------------------------
=== Bai 5 ===
Ma trận bậc thang kết quả:
[0.714285714285714, 1.0*x_3 + 1.14285714285714, x_3]
------------------------------
=== Bai 6 ===
Ma trận bậc thang kết quả:
[3.00000000000000, 13.0000000000000, 9.00000000000000]
------------------------------
=== Bai 7 ===
Ma trận bậc thang kết quả:
[0.5*x_2 + 0.833333333333334*x_4 + 0.5, x_2, 1.33333333333333*x_4 + 0.25, x_4]
------------------------------
=== Bai 8 ===
Ma trận bậc thang kết quả:
[-5.00000000000000, 5.00000000000000, 4.00000000000000]
------------------------